In [2]:
%%writefile transacoes.csv
id,data,cliente_id,tipo,valor,descricao,categoria
1,2026-01-05,CLI001,credito,3500.00,Salário janeiro,salario
2,2026-01-12,CLI002,debito,180.50,Supermercado,compra
3,2026-01-15,CLI004,debito,450.00,Manutenção carro,servico
4,2026-01-22,CLI005,credito,12500.00,Venda de ativo,investimento
5,2026-02-02,CLI001,debito,320.00,Conta de energia,conta
6,2026-02-14,CLI003,credito,15000.00,Transferência suspeita,transferencia
7,2026-02-18,CLI002,debito,85.00,Farmácia,saude
8,2026-02-25,CLI006,debito,1200.00,Aluguel,moradia
9,2026-03-01,CLI001,credito,3500.00,Salário março,salario
10,2026-03-08,CLI007,debito,250.00,Restaurante,alimentacao
11,2026-03-15,CLI002,debito,99.90,Streaming,assinatura
12,2026-03-28,CLI008,debito,410.00,Combustível,transporte
13,2026-04-02,CLI001,credito,4000.00,Adiantamento,salario
14,2026-04-10,CLI009,debito,150.00,Livraria,educacao
15,2026-04-18,CLI003,debito,800.00,Equipamento de rede,ti
16,2026-01-20,CLI001,debito,abc,Erro texto no valor,compra
17,2026-02-03,,debito,200.00,Cliente vazio,transferencia
,2026-02-10,CLI002,credito,500.00,ID vazio,deposito
19,10-03-2026,CLI003,debito,120.00,Data formato BR,compra
20,2026-03-22,CLI004,outro,300.00,Tipo inválido,ajuste
21,2026-04-05,CLI005,debito,-50.00,Valor negativo,estorno

Writing transacoes.csv


In [3]:
import csv
import json
from datetime import datetime
import matplotlib.pyplot as plt

ARQUIVO_ENTRADA = "transacoes.csv"
ARQUIVO_SAIDA_JSON = "relatorio.json"
ARQUIVO_GRAFICO = "grafico.png"
LIMITE_SUSPEITO = 10000.00

def validar_transacao(linha: dict) -> dict | None:
    """Valida cada linha do CSV descartando inconsistências silenciosamente."""
    # Validação do ID
    try:
        id_str = linha.get("id", "").strip()
        if not id_str:
            return None
        transacao_id = int(id_str)
    except (ValueError, TypeError):
        return None

    # Validação do cliente_id
    cliente_id = linha.get("cliente_id", "").strip()
    if not cliente_id:
        return None

    # Validação do tipo
    tipo = linha.get("tipo", "").strip().lower()
    if tipo not in ("credito", "debito"):
        return None

    # Validação do valor (try/except 1)
    try:
        valor_str = linha.get("valor", "").strip()
        valor = float(valor_str)
        if valor <= 0:
            return None
    except (ValueError, TypeError):
        return None

    # Validação da data (try/except 2)
    try:
        data_str = linha.get("data", "").strip()
        data_obj = datetime.strptime(data_str, "%Y-%m-%d")
    except (ValueError, TypeError):
        return None

    return {
        "id": transacao_id,
        "data": data_obj,
        "cliente_id": cliente_id,
        "tipo": tipo,
        "valor": valor,
        "descricao": linha.get("descricao", "").strip(),
        "categoria": linha.get("categoria", "").strip(),
    }

def ler_transacoes(caminho_arquivo: str) -> tuple[list[dict], int, int]:
    """Lê o arquivo CSV via DictReader tratando ausência do arquivo (try/except 3)."""
    transacoes_validas = []
    linhas_invalidas = 0
    total_lidas = 0

    try:
        with open(caminho_arquivo, mode="r", encoding="utf-8") as arquivo:
            leitor = csv.DictReader(arquivo)
            for linha in leitor:
                total_lidas += 1
                registro_limpo = validar_transacao(linha)
                if registro_limpo:
                    transacoes_validas.append(registro_limpo)
                else:
                    linhas_invalidas += 1
    except FileNotFoundError:
        print(f"Erro: O arquivo '{caminho_arquivo}' não foi encontrado.")
        return [], 0, 0

    return transacoes_validas, total_lidas, linhas_invalidas

def gerar_relatorio(transacoes: list[dict], total_lidas: int, total_invalidas: int) -> dict:
    """Calcula as métricas mensais, período e separa suspeitas."""
    if not transacoes:
        return {}

    transacoes_ordenadas = sorted(transacoes, key=lambda t: t["data"])
    data_mais_antiga = transacoes_ordenadas[0]["data"]
    data_mais_recente = transacoes_ordenadas[-1]["data"]
    dias_periodo = (data_mais_recente - data_mais_antiga).days

    resumo_mensal = {}
    transacoes_suspeitas = []

    for t in transacoes:
        mes = t["data"].strftime("%Y-%m")
        valor = t["valor"]
        tipo = t["tipo"]

        if mes not in resumo_mensal:
            resumo_mensal[mes] = {
                "quantidade": 0,
                "total_credito": 0.0,
                "total_debito": 0.0,
                "saldo": 0.0,
                "valores": [],
            }

        resumo_mensal[mes]["quantidade"] += 1
        if tipo == "credito":
            resumo_mensal[mes]["total_credito"] += valor
        else:
            resumo_mensal[mes]["total_debito"] += valor

        resumo_mensal[mes]["valores"].append(valor)

        if valor > LIMITE_SUSPEITO:
            transacoes_suspeitas.append({
                "id": t["id"],
                "cliente_id": t["cliente_id"],
                "data": t["data"].strftime("%Y-%m-%d"),
                "valor": round(valor, 2),
            })

    for mes, dados in resumo_mensal.items():
        dados["total_credito"] = round(dados["total_credito"], 2)
        dados["total_debito"] = round(dados["total_debito"], 2)
        dados["saldo"] = round(dados["total_credito"] - dados["total_debito"], 2)
        dados["media"] = round(sum(dados["valores"]) / dados["quantidade"], 2)
        dados["maior_valor"] = round(max(dados["valores"]), 2)
        dados["menor_valor"] = round(min(dados["valores"]), 2)
        del dados["valores"]

    return {
        "gerado_em": datetime.now().strftime("%Y-%m-%d"),
        "total_linhas_lidas": total_lidas,
        "total_transacoes_validas": len(transacoes),
        "total_transacoes_invalidas": total_invalidas,
        "periodo": {
            "mais_antiga": data_mais_antiga.strftime("%Y-%m-%d"),
            "mais_recente": data_mais_recente.strftime("%Y-%m-%d"),
            "dias_totais": dias_periodo,
        },
        "resumo_mensal": dict(sorted(resumo_mensal.items())),
        "transacoes_suspeitas": transacoes_suspeitas,
    }

def salvar_json(dados_relatorio: dict, caminho_saida: str) -> None:
    """Exporta o relatório consolidado para JSON."""
    with open(caminho_saida, mode="w", encoding="utf-8") as arq:
        json.dump(dados_relatorio, arq, ensure_ascii=False, indent=2)

def formatar_moeda(valor: float) -> str:
    """Converte valor numérico para padrão BRL."""
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

def exibir_relatorio(relatorio: dict) -> None:
    """Imprime o relatório formatado na saída do terminal/notebook."""
    if not relatorio:
        print("Nenhum dado para exibir.")
        return

    print("=" * 45)
    print("        RELATÓRIO DE AUDITORIA CLEARBANK     ")
    print("=" * 45)
    print(f"Gerado em: {relatorio['gerado_em']}")
    print(f"Total de linhas lidas:     {relatorio['total_linhas_lidas']}")
    print(f"Linhas válidas:            {relatorio['total_transacoes_validas']}")
    print(f"Linhas inválidas:          {relatorio['total_transacoes_invalidas']}")

    periodo = relatorio["periodo"]
    print(f"Período analisado:         {periodo['mais_antiga']} → {periodo['mais_recente']} ({periodo['dias_totais']} dias)\n")

    print("===== RELATÓRIO MENSAL =====")
    for mes, m in relatorio["resumo_mensal"].items():
        print(f"Mês: {mes}")
        print(f"  Transações:    {m['quantidade']}")
        print(f"  Total crédito: {formatar_moeda(m['total_credito'])}")
        print(f"  Total débito:  {formatar_moeda(m['total_debito'])}")
        print(f"  Saldo:         {formatar_moeda(m['saldo'])}")
        print(f"  Média:         {formatar_moeda(m['media'])}")
        print(f"  Maior valor:   {formatar_moeda(m['maior_valor'])}")
        print(f"  Menor valor:   {formatar_moeda(m['menor_valor'])}")
        print("-" * 30)

    print("\n===== TRANSAÇÕES SUSPEITAS =====")
    suspeitas = relatorio["transacoes_suspeitas"]
    if suspeitas:
        for s in suspeitas:
            print(f"ID: {s['id']} | Cliente: {s['cliente_id']} | Data: {s['data']} | Valor: {formatar_moeda(s['valor'])}")
    else:
        print("Nenhuma transação suspeita encontrada.")
    print("=" * 45)

def gerar_grafico(relatorio: dict, caminho_img: str) -> None:
    """Gera e salva o gráfico com matplotlib."""
    meses = list(relatorio["resumo_mensal"].keys())
    saldos = [relatorio["resumo_mensal"][m]["saldo"] for m in meses]

    plt.figure(figsize=(8, 4.5))
    cores = ["#2b8a3e" if s >= 0 else "#c92a2a" for s in saldos]
    plt.bar(meses, saldos, color=cores, width=0.5)

    plt.title("Evolução do Saldo Mensal Líquido (Crédito - Débito)", fontsize=13, pad=12)
    plt.xlabel("Mês (AAAA-MM)", fontsize=10)
    plt.ylabel("Saldo (R$)", fontsize=10)
    plt.axhline(0, color="gray", linewidth=0.8, linestyle="--")
    plt.grid(axis="y", linestyle=":", alpha=0.6)
    plt.tight_layout()
    plt.savefig(caminho_img, dpi=300)
    plt.close()
    print(f"\n[OK] Gráfico salvo com sucesso em '{caminho_img}'.")

In [4]:
# Executa o pipeline de ponta a ponta
transacoes, total_lidas, total_invalidas = ler_transacoes(ARQUIVO_ENTRADA)
relatorio_final = gerar_relatorio(transacoes, total_lidas, total_invalidas)
salvar_json(relatorio_final, ARQUIVO_SAIDA_JSON)
exibir_relatorio(relatorio_final)
gerar_grafico(relatorio_final, ARQUIVO_GRAFICO)

        RELATÓRIO DE AUDITORIA CLEARBANK     
Gerado em: 2026-09-03
Total de linhas lidas:     21
Linhas válidas:            15
Linhas inválidas:          6
Período analisado:         2026-01-05 → 2026-04-18 (103 dias)

===== RELATÓRIO MENSAL =====
Mês: 2026-01
  Transações:    4
  Total crédito: R$ 16.000,00
  Total débito:  R$ 630,50
  Saldo:         R$ 15.369,50
  Média:         R$ 4.157,62
  Maior valor:   R$ 12.500,00
  Menor valor:   R$ 180,50
------------------------------
Mês: 2026-02
  Transações:    4
  Total crédito: R$ 15.000,00
  Total débito:  R$ 1.605,00
  Saldo:         R$ 13.395,00
  Média:         R$ 4.151,25
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 85,00
------------------------------
Mês: 2026-03
  Transações:    4
  Total crédito: R$ 3.500,00
  Total débito:  R$ 759,90
  Saldo:         R$ 2.740,10
  Média:         R$ 1.064,97
  Maior valor:   R$ 3.500,00
  Menor valor:   R$ 99,90
------------------------------
Mês: 2026-04
  Transações:    3
  Total crédit